In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import requests
from bs4 import BeautifulSoup
import re

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [3]:

BASE_DIR = Path.cwd()
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR = BASE_DIR / "outputs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Raw data folder: ", RAW_DIR)
print("Processed data folder: ", PROCESSED_DIR)
print("Output folder: ", OUTPUT_DIR)

Raw data folder:  C:\Users\Admin\PycharmProjects\SPU-TEAM-DIRISA\data\raw
Processed data folder:  C:\Users\Admin\PycharmProjects\SPU-TEAM-DIRISA\data\processed
Output folder:  C:\Users\Admin\PycharmProjects\SPU-TEAM-DIRISA\outputs


In [4]:

IEC_URL = "https://www.elections.org.za/pw/StatsData/Voter-Registration-Statistics"
TARGET_PROVINCE = "KwaZulu-Natal"
KZN_PROVINCE_ID = "4"


session = requests.Session()
response = session.get(IEC_URL, timeout=10)

print(f"HTTP status: {response.status_code}")
print(f"Content type: {response.headers.get('content-type')}")
print(f"Response size: {len(response.content)} bytes")
print("IEC source loaded successfully.")
print("Session established successfully.")

HTTP status: 200
Content type: text/html; charset=utf-8
Response size: 131362 bytes
IEC source loaded successfully.
Session established successfully.


In [5]:
# Extract form fields and select province
def extract_form_fields(html):

    soup = BeautifulSoup(html, 'html.parser')
    fields = {}
    for input_field in soup.find_all('input', {'type': 'hidden'}):
        name = input_field.get('name')
        value = input_field.get('value')
        if name and value:
            fields[name] = value
    return fields

def extract_municipalities(html):

    soup = BeautifulSoup(html, 'html.parser')
    municipalities = {}
    municipality_select = soup.find('select', {'id': re.compile('.*Municipality.*', re.I)})
    if municipality_select:
        for option in municipality_select.find_all('option'):
            value = option.get('value')
            text = option.get_text(strip=True)
            if value and text and text != '-- Select Municipality --':
                municipalities[text] = value
    return municipalities


print("Step 1: Extracting ASP.NET form fields...")
form_fields = extract_form_fields(response.text)
print(f"Found {len(form_fields)} form fields")


print("\nStep 2: Selecting KwaZulu-Natal province...")
post_data = {
    **form_fields,
    '__EVENTTARGET': 'ddlProvince',
    '__EVENTARGUMENT': '',
    'ddlProvince': KZN_PROVINCE_ID,
}


response = session.post(IEC_URL, data=post_data)
print(f"Province selection response: {response.status_code}")


form_fields = extract_form_fields(response.text)


print("\nStep 3: Extracting KwaZulu-Natal municipalities...")
municipalities = extract_municipalities(response.text)
print(f"  ✓ Found {len(municipalities)} municipalities:")
for name, value in list(municipalities.items())[:5]:
    print(f"    - {name} (ID: {value})")
if len(municipalities) > 5:
    print(f"    ... and {len(municipalities) - 5} more")

municipality_list = municipalities
print(f"\n Ready to extract wards for {len(municipality_list)} municipalities")

Step 1: Extracting ASP.NET form fields...
  ✓ Found 3 form fields

Step 2: Selecting KwaZulu-Natal province...
  ✓ Province selection response: 200

Step 3: Extracting KwaZulu-Natal municipalities...
  ✓ Found 0 municipalities:

✓ Ready to extract wards for 0 municipalities


In [9]:
#  IEC 2016 data
iec2016_file = RAW_DIR / "IEC_2016.CSV"

if iec2016_file.exists():
    iec2016 = pd.read_csv(iec2016_file, low_memory=False)
    print(f"✓ Loaded IEC 2016 data")
    print(f"  Shape: {iec2016.shape}")
    print(f"  Columns: {list(iec2016.columns)}")
    print("\nFirst few rows:")
    iec2016.head()
else:
    print(f"File not found: {iec2016_file}")
    print("  Run extraction cells above to generate the data file.")

File not found: C:\Users\Admin\PycharmProjects\SPU-TEAM-DIRISA\data\raw\IEC_2016.CSV
  Run extraction cells above to generate the data file.
